# Yakit Ekonomisi ve Filo Stratejisi (Analiz 7)

**Tarih:** 2026-05-11
**Amac:** Yakit turune bagli operasyonel arıza riski + filo yakit maliyeti + modernizasyon senaryosu

**KISIM A - Operasyonel Analiz:** YAKITTURU x ariza karsılastırması, confounder kontrolu
**KISIM B - Finansal Simulasyon:** Mevcut filo yıllık yakıt maliyeti
**KISIM C - Senaryolar:** CNG gecisi ROI hesabı
**KISIM D - Synthesis:** Yenileme oncelik listesi + ML feature + kısıt

---

## Dis Veri Kaynakları
- **Yakit fiyatları:** EPDK aylık ortalama bayi satıs fiyatları 2025 H1, BOTAS
- **Tuketim:** Endustri standardı + Istanbul BRT kalibrasyonu (60 L/100km gercek)
- **Arac fiyatları:** Ankara 2024 ihalesi (Otokar/Mercedes/BMC ~200K EUR solo)

---

## Veri Notu
- ADALAR garajı bilerek dıslandı (golf araclar, farklı operasyonel doga)
- ariza_model.csv 12 garaj iceriyor, ADALAR yok (kontrol edildi)
- ARACTIPI ayrımı yapılmıyor (proje karari, sure kısıtı)
- Yakıt turu "Bilinmiyor" araclar MOTORIN varsayılır (AKIA ULTRA LF12 diger kayıtları motorin)


---
## 1. Veri Yukleme ve Dis Veri Tablosu

Mevcut filodaki marka × model × yakıt × araç tipi profili. Tuketim ve fiyat tabloları dıs veri olarak elle kodlandı.


In [1]:
# BOLUM 1: Veri Yukleme + Dis Veri Tablosu
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv', low_memory=False)
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
print(f'Ariza: {len(df):,} kayit, {df["KAPINO"].nunique():,} arac')

# Bilinmiyor YAKITTURU duzeltme: MODEL adinda 'CNG' geciyorsa CNG, yoksa MOTORIN
mask_bilinmiyor = df['YAKITTURU']=='Bilinmiyor'
print(f'YAKITTURU Bilinmiyor (duzeltme oncesi): {mask_bilinmiyor.sum():,} kayit')
mask_cng_model = df['MODEL'].str.contains('CNG', na=False)
df.loc[mask_bilinmiyor & mask_cng_model, 'YAKITTURU'] = 'CNG'
df.loc[mask_bilinmiyor & ~mask_cng_model, 'YAKITTURU'] = 'MOTORIN'
print(f'  Bilinmiyor -> CNG (MODEL adinda CNG var):       {(mask_bilinmiyor & mask_cng_model).sum():,} kayit')
print(f'  Bilinmiyor -> MOTORIN (MODEL adinda CNG yok):   {(mask_bilinmiyor & ~mask_cng_model).sum():,} kayit')
# Elektrik 2 kayit (Citaro elektrik) ve E-JEST 2 kayit ihmal edilebilir
df = df[df['YAKITTURU'].isin(['MOTORIN','CNG'])].copy()
print(f'Filtre sonrasi: {len(df):,} kayit, {df["KAPINO"].nunique():,} arac')

# DIS VERI 1: TUKETIM TABLOSU (L/100km motorin, m3/100km CNG)
# Endustri standardi + Istanbul BRT kalibrasyonu (60 L/100km koruklu ortalama)
tuketim = pd.DataFrame([
    ('OTOKAR',   'KENT 290LF',     'SOLO',    'MOTORIN', 40.0),
    ('OTOKAR',   'KENT XL',        'KORUKLU', 'MOTORIN', 60.0),
    ('MERCEDES', 'CITARO 0530',    'SOLO',    'MOTORIN', 39.0),
    ('MERCEDES', 'CITARO 0530 G',  'KORUKLU', 'MOTORIN', 58.0),
    ('MERCEDES', 'CONECTO G',      'KORUKLU', 'MOTORIN', 62.0),
    ('MERCEDES', 'CONECTO',        'SOLO',    'MOTORIN', 42.0),
    ('MERCEDES', 'CAPACITY',       'KORUKLU', 'MOTORIN', 65.0),
    ('BMC',      'PROCITY TR',     'SOLO',    'MOTORIN', 41.0),
    ('BMC',      'PROCITY',        'SOLO',    'MOTORIN', 41.0),
    ('KARSAN',   'AVANCITY S PLUS','KORUKLU', 'MOTORIN', 58.0),
    ('KARSAN',   'AVANCITY CNG',   'SOLO',    'CNG',     52.0),
    ('TEMSA',    'AVENUE LF CNG',  'SOLO',    'CNG',     50.0),
    ('AKIA',     'ULTRA LF12',     'SOLO',    'MOTORIN', 40.0),
    ('AKIA',     'LF25',           'KORUKLU', 'MOTORIN', 60.0),
], columns=['MARKA','MODEL','ARACCINSI','YAKITTURU','TUKETIM_BIRIM_100KM'])
print(f'\nTUKETIM TABLOSU ({len(tuketim)} marka-model):')
print(tuketim.to_string(index=False))

# DIS VERI 2: YAKIT FIYATLARI 2025 H1
# Motorin: EPDK aylık ortalama bayi satıs fiyatları
# CNG: BOTAS / belediye tarifeleri ortalama
yakit_fiyat = pd.DataFrame([
    ('2025-01', 46.29, 20.77),
    ('2025-02', 46.77, 20.77),
    ('2025-03', 45.92, 20.77),
    ('2025-04', 45.26, 20.77),
    ('2025-05', 45.45, 20.77),
    ('2025-06', 48.81, 20.77),
], columns=['AY','MOTORIN_TL_L','CNG_TL_M3'])
yakit_fiyat['H1_ort_motorin'] = yakit_fiyat['MOTORIN_TL_L'].mean()
yakit_fiyat['H1_ort_cng'] = yakit_fiyat['CNG_TL_M3'].mean()
print(f'\nYAKIT FIYAT 2025 H1:')
print(yakit_fiyat[['AY','MOTORIN_TL_L','CNG_TL_M3']].to_string(index=False))
print(f'\nH1 ortalama motorin: {yakit_fiyat["H1_ort_motorin"].iloc[0]:.2f} TL/L')
print(f'H1 ortalama CNG:     {yakit_fiyat["H1_ort_cng"].iloc[0]:.2f} TL/m3')

MOTORIN_FIYAT = yakit_fiyat['H1_ort_motorin'].iloc[0]  # TL/L
CNG_FIYAT = yakit_fiyat['H1_ort_cng'].iloc[0]          # TL/m3

# DIS VERI 3: ARAC ALIM FIYATLARI (Ankara 2024 ihalesi + endustri std)
arac_fiyat = pd.DataFrame([
    ('SOLO',    'MOTORIN',  200_000),  # Otokar Ankara: 5.73M / 28 = 204,643
    ('SOLO',    'CNG',      200_000),  # Mercedes Ankara: 51.32M / 254 = 202,047
    ('KORUKLU', 'MOTORIN',  360_000),  # Solo +%80 endustri std
    ('KORUKLU', 'CNG',      380_000),
], columns=['ARACCINSI','YAKITTURU','ALIM_FIYAT_EUR'])
EUR_TRY = 35.0  # 2025 H1 ortalama kur tahmini
arac_fiyat['ALIM_FIYAT_TL'] = arac_fiyat['ALIM_FIYAT_EUR'] * EUR_TRY
print(f'\nARAC ALIM FIYATLARI (Ankara 2024 ihalesi + endustri std):')
print(arac_fiyat.to_string(index=False))


Ariza: 58,559 kayit, 3,509 arac
YAKITTURU Bilinmiyor (duzeltme oncesi): 640 kayit
  Bilinmiyor -> CNG (MODEL adinda CNG var):       48 kayit
  Bilinmiyor -> MOTORIN (MODEL adinda CNG yok):   592 kayit
Filtre sonrasi: 58,557 kayit, 3,508 arac

TUKETIM TABLOSU (14 marka-model):
   MARKA           MODEL ARACCINSI YAKITTURU  TUKETIM_BIRIM_100KM
  OTOKAR      KENT 290LF      SOLO   MOTORIN                 40.0
  OTOKAR         KENT XL   KORUKLU   MOTORIN                 60.0
MERCEDES     CITARO 0530      SOLO   MOTORIN                 39.0
MERCEDES   CITARO 0530 G   KORUKLU   MOTORIN                 58.0
MERCEDES       CONECTO G   KORUKLU   MOTORIN                 62.0
MERCEDES         CONECTO      SOLO   MOTORIN                 42.0
MERCEDES        CAPACITY   KORUKLU   MOTORIN                 65.0
     BMC      PROCITY TR      SOLO   MOTORIN                 41.0
     BMC         PROCITY      SOLO   MOTORIN                 41.0
  KARSAN AVANCITY S PLUS   KORUKLU   MOTORIN                 58

---
## 2. KISIM A: YAKITTURU x Ariza Temel Karsılastırma

Soru: CNG vs MOTORIN aracları ciddi arıza riski farklı mı?
Yontem: Chi-square + lift hesabı + ciddi_oran karsilastirmasi


In [2]:
# BOLUM 2: YAKITTURU x ciddi_ariza
from scipy.stats import chi2_contingency

# Yakit turu basina arac sayisi ve ciddi oran
ozet = df.groupby('YAKITTURU').agg(
    n_ariza=('ciddi_ariza','count'),
    n_arac=('KAPINO','nunique'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_skor=('ciddiyet_skoru','mean'),
).round(3)
print('=== YAKITTURU OZETI ===')
print(ozet.to_string())

# Chi-square
ct = pd.crosstab(df['YAKITTURU'], df['ciddi_ariza'])
print(f'\n=== CHI-SQUARE TEST ===')
print(ct)
chi2, p_chi, dof, _ = chi2_contingency(ct)
print(f'chi2={chi2:.2f}, p={p_chi:.6e}, dof={dof}')

# Lift
genel_ciddi = df['ciddi_ariza'].mean()
print(f'\n=== LIFT (referans = genel ciddi oran {genel_ciddi:.3f}) ===')
for yt in ['MOTORIN','CNG']:
    sub = df[df['YAKITTURU']==yt]
    oran = sub['ciddi_ariza'].mean()
    lift = oran / genel_ciddi
    print(f'  {yt}: ciddi_oran={oran:.3f}, lift={lift:.3f}')

# Yorum
motorin_ciddi = df[df['YAKITTURU']=='MOTORIN']['ciddi_ariza'].mean()
cng_ciddi = df[df['YAKITTURU']=='CNG']['ciddi_ariza'].mean()
fark = (motorin_ciddi - cng_ciddi) / motorin_ciddi * 100
print(f'\n=== YORUM ===')
print(f'CNG aracları motorinden %{fark:.1f} daha az ciddi ariza yapıyor (kontrolsuz)')
print(f'Bu fark confounder olabilir (CNG araclar yeni olabilir) - Bolum 4 kontrolu')


=== YAKITTURU OZETI ===
           n_ariza  n_arac  ciddi_oran  ort_skor
YAKITTURU                                       
CNG           4825     350       0.284     3.521
MOTORIN      53732    3158       0.389     3.618

=== CHI-SQUARE TEST ===
ciddi_ariza      0      1
YAKITTURU                
CNG           3453   1372
MOTORIN      32845  20887
chi2=204.25, p=2.470629e-46, dof=1

=== LIFT (referans = genel ciddi oran 0.380) ===
  MOTORIN: ciddi_oran=0.389, lift=1.023
  CNG: ciddi_oran=0.284, lift=0.748

=== YORUM ===
CNG aracları motorinden %26.9 daha az ciddi ariza yapıyor (kontrolsuz)
Bu fark confounder olabilir (CNG araclar yeni olabilir) - Bolum 4 kontrolu


---
## 3. EMISYON x Ariza Karsılastırması

Soru: Emisyon standardı (EEV/EUR/Euro1) ariza riski ile iliskili mi?


In [3]:
# BOLUM 3: EMISYON x ciddi_ariza
ozet_em = df.groupby('EMISYON').agg(
    n_ariza=('ciddi_ariza','count'),
    n_arac=('KAPINO','nunique'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_skor=('ciddiyet_skoru','mean'),
).round(3).sort_values('ciddi_oran')
print('=== EMISYON OZETI ===')
print(ozet_em.to_string())

# Chi-square (Bilinmiyor haric)
df_em = df[df['EMISYON'].isin(['EEV','EUR','1'])]
ct = pd.crosstab(df_em['EMISYON'], df_em['ciddi_ariza'])
chi2, p_chi, dof, _ = chi2_contingency(ct)
print(f'\nChi-square (EEV vs EUR vs 1): chi2={chi2:.2f}, p={p_chi:.6e}')

# YAKITTURU x EMISYON capraz
print(f'\n=== YAKITTURU x EMISYON capraz (arac sayisi) ===')
arac_uniq = df.groupby('KAPINO').agg(YAKITTURU=('YAKITTURU','first'), EMISYON=('EMISYON','first')).reset_index()
print(pd.crosstab(arac_uniq['YAKITTURU'], arac_uniq['EMISYON']).to_string())

print(f'\n=== YORUM ===')
print('EEV-EUR ciddi_oran farkı cok kucuk (genelde <0.01)')
print('Yas confounder iyice yakalanir')


=== EMISYON OZETI ===
            n_ariza  n_arac  ciddi_oran  ort_skor
EMISYON                                          
1                29       1       0.241     3.191
EEV           26683    1571       0.374     3.609
EUR           26021    1516       0.382     3.611
Bilinmiyor     5824     420       0.400     3.610

Chi-square (EEV vs EUR vs 1): chi2=6.40, p=4.078403e-02

=== YAKITTURU x EMISYON capraz (arac sayisi) ===
EMISYON    1  Bilinmiyor   EEV   EUR
YAKITTURU                           
CNG        1          14   335     0
MOTORIN    0         406  1236  1516

=== YORUM ===
EEV-EUR ciddi_oran farkı cok kucuk (genelde <0.01)
Yas confounder iyice yakalanir


---
## 4. Confounder Kontrolu: CNG Avantajı Gercek mi?

Soru: CNG araclar yeni mi (yas confounder) yoksa gercekten daha az arıza mı?
Yontem: Multiple regression M1->M4 (yakıt + yas + garaj + arac cinsi)


In [4]:
# BOLUM 4: Confounder kontrolu
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Arac bazinda profil
arac_yk = df.groupby('KAPINO').agg(
    YAKITTURU=('YAKITTURU','first'),
    MARKA=('MARKA','first'),
    MODEL=('MODEL','first'),
    MODELYILI=('MODELYILI','first'),
    ARACCINSI=('ARACCINSI','first'),
    GARAJ=('GARAJ','first'),
    EMISYON=('EMISYON','first'),
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi_oran=('ciddi_ariza','mean'),
    n_ariza=('ciddi_ariza','count'),
).reset_index()
arac_yk['yas'] = 2025 - arac_yk['MODELYILI']
arac_yk = arac_yk.dropna(subset=['yas','GARAJ','YAKITTURU','ARACCINSI']).copy()
arac_yk = arac_yk[arac_yk['YAKITTURU'].isin(['MOTORIN','CNG'])]
arac_yk['is_cng'] = (arac_yk['YAKITTURU']=='CNG').astype(int)
print(f'Regresyon seti: {len(arac_yk):,} arac')

# Yas dagilimi yakit turune gore
print(f'\n=== YAS PROFILI ===')
print(arac_yk.groupby('YAKITTURU')['yas'].describe().round(1).to_string())

# Modeller (bagimli: ort_skor)
m1 = ols('ort_skor ~ is_cng', data=arac_yk).fit()
m2 = ols('ort_skor ~ is_cng + yas', data=arac_yk).fit()
m3 = ols('ort_skor ~ is_cng + yas + C(ARACCINSI)', data=arac_yk).fit()
m4 = ols('ort_skor ~ is_cng + yas + C(ARACCINSI) + C(GARAJ)', data=arac_yk).fit()

print(f'\n=== R^2 + CNG KATSAYISI ===')
print(f'M1 (sadece is_cng):                R^2={m1.rsquared:.4f}, k_cng={m1.params["is_cng"]:+.4f}, p={m1.pvalues["is_cng"]:.4f}')
print(f'M2 (+yas):                         R^2={m2.rsquared:.4f}, k_cng={m2.params["is_cng"]:+.4f}, p={m2.pvalues["is_cng"]:.4f}')
print(f'M3 (+yas+aractipi):                R^2={m3.rsquared:.4f}, k_cng={m3.params["is_cng"]:+.4f}, p={m3.pvalues["is_cng"]:.4f}')
print(f'M4 (+yas+aractipi+garaj):          R^2={m4.rsquared:.4f}, k_cng={m4.params["is_cng"]:+.4f}, p={m4.pvalues["is_cng"]:.4f}')

# Yorum
k_m1 = m1.params['is_cng']
k_m4 = m4.params['is_cng']
print(f'\n=== YORUM ===')
print(f'CNG katsayisi M1={k_m1:+.4f} -> M4={k_m4:+.4f}')
if abs(k_m4) < abs(k_m1) * 0.5:
    print(f'  CNG etkisi M4`te buyuk olcude DUSTU - cogu yas/garaj/aractipi confounder')
elif abs(k_m4) < abs(k_m1) * 0.8:
    print(f'  CNG etkisi kismi confounder altında ama bagimsiz sinyal var')
else:
    print(f'  CNG etkisi confounder altında KORUNUYOR - gercek yakıt turu avantajı')


Regresyon seti: 3,508 arac

=== YAS PROFILI ===
            count  mean  std   min   25%   50%   75%   max
YAKITTURU                                                 
CNG         350.0  11.7  0.5  10.0  11.0  12.0  12.0  12.0
MOTORIN    3158.0  11.5  4.9   1.0   9.0  12.0  13.0  19.0

=== R^2 + CNG KATSAYISI ===
M1 (sadece is_cng):                R^2=0.0013, k_cng=+0.0884, p=0.0355
M2 (+yas):                         R^2=0.0449, k_cng=+0.0835, p=0.0423
M3 (+yas+aractipi):                R^2=0.0839, k_cng=+0.2116, p=0.0000
M4 (+yas+aractipi+garaj):          R^2=0.2496, k_cng=-0.3575, p=0.0000

=== YORUM ===
CNG katsayisi M1=+0.0884 -> M4=-0.3575
  CNG etkisi confounder altında KORUNUYOR - gercek yakıt turu avantajı


---
## 5. MARKA x YAKITTURU Lift Matrisi

Soru: CNG avantajı tum markalar icin gecerli mi, yoksa belirli markalara mı ozgu?


In [5]:
# BOLUM 5: MARKA x YAKITTURU lift
genel_ciddi = df['ciddi_ariza'].mean()

# MARKA x YAKITTURU lift matrix
matris = df.groupby(['MARKA','YAKITTURU']).agg(
    n_ariza=('ciddi_ariza','count'),
    ciddi_oran=('ciddi_ariza','mean'),
).reset_index()
matris['lift'] = (matris['ciddi_oran'] / genel_ciddi).round(3)
matris = matris.sort_values(['MARKA','YAKITTURU'])
print('=== MARKA x YAKITTURU LIFT ===')
print(matris.to_string(index=False))

# MARKA bazinda CNG/MOTORIN karsilastirma
print(f'\n=== MARKA ICINDE CNG vs MOTORIN ===')
for marka in matris['MARKA'].unique():
    sub = matris[matris['MARKA']==marka]
    yakitlar = sub['YAKITTURU'].unique()
    if 'CNG' in yakitlar and 'MOTORIN' in yakitlar:
        m_oran = sub[sub['YAKITTURU']=='MOTORIN']['ciddi_oran'].iloc[0]
        c_oran = sub[sub['YAKITTURU']=='CNG']['ciddi_oran'].iloc[0]
        fark = (m_oran - c_oran) / m_oran * 100
        print(f'  {marka}: MOTORIN={m_oran:.3f}, CNG={c_oran:.3f}, CNG avantajı %{fark:+.1f}')

print(f'\n=== YORUM ===')
print('Sadece KARSAN ve TEMSA filoda CNG araca sahip')
print('Bu markaların kendi MOTORIN aracları ile dogrudan karsilastirma sinirli')


=== MARKA x YAKITTURU LIFT ===
   MARKA YAKITTURU  n_ariza  ciddi_oran  lift
    AKIA   MOTORIN     2572    0.328538 0.864
     BMC   MOTORIN     4675    0.361070 0.950
  KARSAN       CNG     2294    0.333915 0.878
  KARSAN   MOTORIN     7629    0.434526 1.143
MERCEDES   MOTORIN    21346    0.387051 1.018
  OTOKAR   MOTORIN    17510    0.387036 1.018
   TEMSA       CNG     2531    0.239431 0.630

=== MARKA ICINDE CNG vs MOTORIN ===
  KARSAN: MOTORIN=0.435, CNG=0.334, CNG avantajı %+23.2

=== YORUM ===
Sadece KARSAN ve TEMSA filoda CNG araca sahip
Bu markaların kendi MOTORIN aracları ile dogrudan karsilastirma sinirli


---
## 6. KISIM B: Filo Yıllık Km ve Yakıt Tuketim Tahmini

Bu bolumde her arac icin yıllık km tahmini cikarip, tuketim tablosu ile birlestiriyoruz.
6 ay veri var -> yıllık ekstrapolasyon (×2)


In [6]:
# BOLUM 6: Arac yillik km tahmini
sefer = pd.read_csv('../panel_data/temiz_veri/sefer_temiz.csv',
                    usecols=['KAPINO','BASLANGICZAMANI','GUZERGAHUZUNLUK','GERCEKLESENGUZERGAHUZUNLUK'],
                    low_memory=False)
sefer['baslangic_dt'] = pd.to_datetime(sefer['BASLANGICZAMANI'], format='mixed', errors='coerce')
sefer = sefer.dropna(subset=['baslangic_dt'])
sefer['km'] = sefer['GERCEKLESENGUZERGAHUZUNLUK'].fillna(sefer['GUZERGAHUZUNLUK']) / 1000
sefer = sefer[(sefer['km'] > 0) & (sefer['km'] < 200)]

# 6 ay -> yıllık (×2)
arac_km = sefer.groupby('KAPINO')['km'].sum().reset_index()
arac_km.columns = ['KAPINO','toplam_km_6ay']
arac_km['yillik_km_tahmini'] = arac_km['toplam_km_6ay'] * 2

print(f'=== ARAC KM (6 ay -> yillik) ===')
print(arac_km['yillik_km_tahmini'].describe().round(0).to_string())

# Arac profili + km
arac_full = arac_yk.merge(arac_km, on='KAPINO', how='left')
arac_full['yillik_km_tahmini'] = arac_full['yillik_km_tahmini'].fillna(arac_full['yillik_km_tahmini'].median())
print(f'\n{len(arac_full):,} arac × yillik km tahmini birlestirildi')

# Tuketim tablosuyla birlestir
arac_full = arac_full.merge(tuketim[['MARKA','MODEL','TUKETIM_BIRIM_100KM']], on=['MARKA','MODEL'], how='left')
print(f'\nTuketim tablosundaki eksik kayit: {arac_full["TUKETIM_BIRIM_100KM"].isna().sum():,}')
# Eksikler icin medyan kullan
arac_full['TUKETIM_BIRIM_100KM'] = arac_full['TUKETIM_BIRIM_100KM'].fillna(arac_full['TUKETIM_BIRIM_100KM'].median())

# Yillik tuketim
arac_full['yillik_tuketim'] = arac_full['yillik_km_tahmini'] * arac_full['TUKETIM_BIRIM_100KM'] / 100
print(f'\n=== YILLIK TUKETIM (L motorin veya m3 CNG) ===')
print(arac_full.groupby('YAKITTURU')['yillik_tuketim'].agg(['mean','median','sum']).round(0).to_string())


=== ARAC KM (6 ay -> yillik) ===
count      6728.0
mean      59206.0
std       24854.0
min          10.0
25%       44125.0
50%       55632.0
75%       72800.0
max      153857.0

3,508 arac × yillik km tahmini birlestirildi

Tuketim tablosundaki eksik kayit: 1

=== YILLIK TUKETIM (L motorin veya m3 CNG) ===
              mean   median         sum
YAKITTURU                              
CNG        19624.0  19542.0   6868257.0
MOTORIN    29231.0  21747.0  92312078.0


---
## 7. Mevcut Filo Yıllık Yakıt Maliyeti

Her arac icin: yillik_km × L/100km × TL/L = yillik yakıt TL


In [7]:
# BOLUM 7: Yakit Maliyeti
arac_full['birim_fiyat'] = arac_full['YAKITTURU'].map({'MOTORIN': MOTORIN_FIYAT, 'CNG': CNG_FIYAT})
arac_full['yillik_yakit_tl'] = arac_full['yillik_tuketim'] * arac_full['birim_fiyat']

# Filo toplam
toplam_tl = arac_full['yillik_yakit_tl'].sum()
ort_tl_arac = arac_full['yillik_yakit_tl'].mean()
print(f'=== FILO YILLIK YAKIT MALIYETI ===')
print(f'TOPLAM: {toplam_tl/1e9:.2f} milyar TL')
print(f'Arac basina ortalama: {ort_tl_arac/1000:.0f} bin TL/yil')

# Yakit turune gore
print(f'\n=== YAKIT TURU BAZINDA ===')
yt_ozet = arac_full.groupby('YAKITTURU').agg(
    n_arac=('KAPINO','count'),
    yillik_km_ort=('yillik_km_tahmini','mean'),
    yillik_yakit_ort=('yillik_yakit_tl','mean'),
    yillik_yakit_top=('yillik_yakit_tl','sum'),
).round(0)
print(yt_ozet.to_string())

# Garaj bazinda dokum
print(f'\n=== GARAJ x YAKIT MALIYETI ===')
gy = arac_full.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_ort=('yas','mean'),
    yillik_yakit_top=('yillik_yakit_tl','sum'),
    yillik_yakit_ort=('yillik_yakit_tl','mean'),
).round(0).sort_values('yillik_yakit_top', ascending=False)
print(gy.to_string())

# Yas bandina gore
print(f'\n=== YAS BANDI x YAKIT MALIYETI ===')
arac_full['yas_band'] = pd.cut(arac_full['yas'], bins=[-1, 3, 10, 15, 100], labels=['Yeni 0-3','Orta 3-10','Yasli 10-15','Cok Yasli 15+'])
yb = arac_full.groupby('yas_band', observed=True).agg(
    n_arac=('KAPINO','count'),
    yillik_yakit_top=('yillik_yakit_tl','sum'),
    yillik_yakit_ort=('yillik_yakit_tl','mean'),
).round(0)
print(yb.to_string())


=== FILO YILLIK YAKIT MALIYETI ===
TOPLAM: 4.43 milyar TL
Arac basina ortalama: 1262 bin TL/yil

=== YAKIT TURU BAZINDA ===
           n_arac  yillik_km_ort  yillik_yakit_ort  yillik_yakit_top
YAKITTURU                                                           
CNG           350        38251.0          407582.0      1.426537e+08
MOTORIN      3158        57457.0         1356814.0      4.284819e+09

=== GARAJ x YAKIT MALIYETI ===
                           n_arac  yas_ort  yillik_yakit_top  yillik_yakit_ort
GARAJ                                                                         
Edirnekapı                    381     12.0      1.044645e+09         2741850.0
Hasanpaşa                     319      8.0      9.443454e+08         2960330.0
IKITELLIGARAJI                429      9.0      4.656352e+08         1085397.0
SULTANGAZIGARAJI              413     12.0      4.312146e+08         1044103.0
KURTKÖY                       346     12.0      2.999851e+08          867009.0
IKITELLIISLETTI

---
## 8. Marka x Model x Maliyet Verimsizlik Sıralaması

Soru: En verimsiz marka-model hangisi? Yıllık yakıt maliyeti / yıllık km?


In [8]:
# BOLUM 8: Verimsizlik sıralaması
verim = arac_full.groupby(['MARKA','MODEL']).agg(
    n_arac=('KAPINO','count'),
    yas_ort=('yas','mean'),
    yillik_km_ort=('yillik_km_tahmini','mean'),
    tuketim_100km=('TUKETIM_BIRIM_100KM','first'),
    yakit_turu=('YAKITTURU','first'),
    yillik_yakit_ort=('yillik_yakit_tl','mean'),
).round(0).reset_index()
verim['maliyet_per_km'] = (verim['yillik_yakit_ort'] / verim['yillik_km_ort']).round(2)
verim = verim.sort_values('maliyet_per_km', ascending=False)
print('=== MARKA x MODEL VERIMSIZLIK SIRALAMASI (TL/km) ===')
print(verim.to_string(index=False))

print(f'\n=== YORUM ===')
print('En yuksek TL/km - en verimsiz model adayi')
print('Yas + tuketim birlikte yorumlanmalı')


=== MARKA x MODEL VERIMSIZLIK SIRALAMASI (TL/km) ===
   MARKA           MODEL  n_arac  yas_ort  yillik_km_ort  tuketim_100km yakit_turu  yillik_yakit_ort  maliyet_per_km
MERCEDES        CAPACITY     249     17.0        99713.0           65.0    MOTORIN         3008413.0           30.17
MERCEDES       CONECTO G     388     12.0        74342.0           62.0    MOTORIN         2139432.0           28.78
    AKIA            LF25     132      2.0        80396.0           60.0    MOTORIN         2239015.0           27.85
  OTOKAR         KENT XL     119      3.0        97974.0           60.0    MOTORIN         2728569.0           27.85
  KARSAN AVANCITY S PLUS     304     12.0        37136.0           58.0    MOTORIN          999773.0           26.92
MERCEDES   CITARO 0530 G      86     19.0        38004.0           58.0    MOTORIN         1023139.0           26.92
MERCEDES         CONECTO      10     13.0        47471.0           42.0    MOTORIN          925453.0           19.50
     BMC   

---
## 9. KISIM C: Senaryo 1 - Top 50 Yaslı Motorin Aracın CNG Gecisi

Plan: En yaslı 50 motorin solo aracı CNG modeline donustur.
Hesap:
- Yakıt tasarrufu = mevcut motorin TL - yeni CNG TL
- Arıza tasarrufu = M4'ten CNG katsayisı (operasyonel)
- Yatırım = 50 × 200K EUR × 35 TL/EUR
- Geri odeme suresi = yatirim / yıllık tasarruf


In [9]:
# BOLUM 9: Senaryo 1 - 50 yaslı motorin -> CNG
solo_motorin = arac_full[(arac_full['YAKITTURU']=='MOTORIN') & (arac_full['ARACCINSI']=='SOLO')].copy()
hedef = solo_motorin.nlargest(50, 'yas').copy()
print(f'=== SENARYO 1: TOP 50 YASLI MOTORIN SOLO ===')
print(f'Hedef arac sayisi: {len(hedef)}')
print(f'Yas dagilimi: ort={hedef["yas"].mean():.1f}, min={hedef["yas"].min()}, max={hedef["yas"].max()}')
print(f'Garaj dagilimi:')
print(hedef['GARAJ'].value_counts())

# Mevcut yıllık maliyet
mevcut_maliyet = hedef['yillik_yakit_tl'].sum()

# CNG donusum sonrası maliyet (KARSAN AVANCITY CNG ortalaması)
CNG_TUK = 52.0  # m3/100km
yeni_maliyet = (hedef['yillik_km_tahmini'] * CNG_TUK / 100 * CNG_FIYAT).sum()

yakit_tasarruf = mevcut_maliyet - yeni_maliyet
print(f'\n=== YAKIT TASARRUF ===')
print(f'Mevcut yillik maliyet:  {mevcut_maliyet/1e6:.2f} milyon TL')
print(f'CNG sonrası maliyet:    {yeni_maliyet/1e6:.2f} milyon TL')
print(f'YILLIK TASARRUF:        {yakit_tasarruf/1e6:.2f} milyon TL')

# Yatirim
YATIRIM_BIRIM_EUR = 200_000
yatirim_eur = 50 * YATIRIM_BIRIM_EUR
yatirim_tl = yatirim_eur * EUR_TRY
print(f'\n=== YATIRIM ===')
print(f'50 arac × {YATIRIM_BIRIM_EUR:,} EUR = {yatirim_eur:,} EUR')
print(f'TL karsiligi (kur 35): {yatirim_tl/1e6:.2f} milyon TL')

# Geri odeme suresi
geri_odeme = yatirim_tl / yakit_tasarruf
print(f'\n=== ROI ===')
print(f'Geri odeme suresi (sadece yakit tasarrufu): {geri_odeme:.1f} yil')

# Arıza tasarrufu (M4 CNG katsayisi etkisi)
k_cng_m4 = m4.params['is_cng']
print(f'\n=== ARIZA TASARRUFU (M4 CNG katsayisi etkisi) ===')
print(f'M4 CNG katsayisi: {k_cng_m4:+.4f} (ort_skor uzerinde)')
mevcut_ciddi_oran = hedef['ciddi_oran'].mean()
cng_ciddi_oran = arac_full[arac_full['YAKITTURU']=='CNG']['ciddi_oran'].mean()
hedef_n_ariza = hedef['n_ariza'].sum()
tasarruf_oran = (mevcut_ciddi_oran - cng_ciddi_oran) / mevcut_ciddi_oran
print(f'Mevcut 50 aracın ortalama ciddi_oran: {mevcut_ciddi_oran:.3f}')
print(f'CNG araclar ciddi_oran: {cng_ciddi_oran:.3f}')
print(f'Tasarruf oranı (kontrolsuz): %{tasarruf_oran*100:.1f}')


=== SENARYO 1: TOP 50 YASLI MOTORIN SOLO ===
Hedef arac sayisi: 50
Yas dagilimi: ort=19.0, min=19.0, max=19.0
Garaj dagilimi:
GARAJ
Anadolu      31
Şahinkaya    19
Name: count, dtype: int64

=== YAKIT TASARRUF ===
Mevcut yillik maliyet:  41.08 milyon TL
CNG sonrası maliyet:    24.51 milyon TL
YILLIK TASARRUF:        16.57 milyon TL

=== YATIRIM ===
50 arac × 200,000 EUR = 10,000,000 EUR
TL karsiligi (kur 35): 350.00 milyon TL

=== ROI ===
Geri odeme suresi (sadece yakit tasarrufu): 21.1 yil

=== ARIZA TASARRUFU (M4 CNG katsayisi etkisi) ===
M4 CNG katsayisi: -0.3575 (ort_skor uzerinde)
Mevcut 50 aracın ortalama ciddi_oran: 0.342
CNG araclar ciddi_oran: 0.298
Tasarruf oranı (kontrolsuz): %12.8


---
## 10. Senaryo 2 - 5 Yıllık Asamali CNG Gecisi

Plan: Her yıl filodaki en yaslı motorin aracın %20'sini CNG'ye donustur.
5 yıllık birikimli tasarruf hesabı.


In [10]:
# BOLUM 10: Senaryo 2 - 5 yillik asamali gecis
n_motorin = (arac_full['YAKITTURU']=='MOTORIN').sum()
yillik_donus = int(n_motorin * 0.20)
print(f'Toplam motorin: {n_motorin}, yıllık donusum: {yillik_donus}')

# 5 yillik birikimli
yillik_tasarruf_list = []
for yil in range(1, 6):
    n_donus = yillik_donus * yil
    if n_donus > n_motorin:
        n_donus = n_motorin
    # En yasli n_donus motorini sec
    motorin_arac = arac_full[arac_full['YAKITTURU']=='MOTORIN'].nlargest(n_donus, 'yas')
    mevcut = motorin_arac['yillik_yakit_tl'].sum()
    cng_alt = (motorin_arac['yillik_km_tahmini'] * CNG_TUK / 100 * CNG_FIYAT).sum()
    tasarruf = mevcut - cng_alt
    yillik_tasarruf_list.append({'Yil': yil, 'Donusturulen': n_donus, 'Yillik_tasarruf_milyon_TL': tasarruf/1e6})

senaryo2_df = pd.DataFrame(yillik_tasarruf_list)
senaryo2_df['Birikimli_tasarruf'] = senaryo2_df['Yillik_tasarruf_milyon_TL'].cumsum()
print(f'\n=== 5 YILLIK ASAMALI GECIS TASARRUFU ===')
print(senaryo2_df.to_string(index=False))

# Toplam 5 yıl yatirim
toplam_donus = yillik_donus * 5
if toplam_donus > n_motorin:
    toplam_donus = n_motorin
toplam_yatirim_tl = toplam_donus * YATIRIM_BIRIM_EUR * EUR_TRY
print(f'\n=== 5 YILLIK TOPLAM ===')
print(f'Toplam donusum: {toplam_donus} arac')
print(f'Toplam yatirim: {toplam_yatirim_tl/1e6:.0f} milyon TL')
print(f'5 yıllık birikimli tasarruf: {senaryo2_df["Birikimli_tasarruf"].iloc[-1]:.0f} milyon TL')


Toplam motorin: 3158, yıllık donusum: 631

=== 5 YILLIK ASAMALI GECIS TASARRUFU ===
 Yil  Donusturulen  Yillik_tasarruf_milyon_TL  Birikimli_tasarruf
   1           631                 527.482053          527.482053
   2          1262                1081.579530         1609.061583
   3          1893                1323.181675         2932.243258
   4          2524                1782.402539         4714.645797
   5          3155                2324.277726         7038.923523

=== 5 YILLIK TOPLAM ===
Toplam donusum: 3155 arac
Toplam yatirim: 22085 milyon TL
5 yıllık birikimli tasarruf: 7039 milyon TL


---
## 11. Sensitivity Analizi - Tahminlerimizin Robustlugu

Senaryolarımız ne kadar guvenli? Tuketim ±%20, fiyat ±%15 varsayımlarıyla test edelim.


In [11]:
# BOLUM 11: Sensitivity Analizi
print('=== SENSITIVITY: SENARYO 1 (50 yasli motorin -> CNG) ===\n')

senaryolar = []
for tuk_carpan in [0.8, 1.0, 1.2]:
    for fiyat_carpan in [0.85, 1.0, 1.15]:
        m_fiyat = MOTORIN_FIYAT * fiyat_carpan
        c_fiyat = CNG_FIYAT * fiyat_carpan
        mev = (hedef['yillik_km_tahmini'] * hedef['TUKETIM_BIRIM_100KM'] * tuk_carpan / 100 * m_fiyat).sum()
        cng_alt = (hedef['yillik_km_tahmini'] * CNG_TUK * tuk_carpan / 100 * c_fiyat).sum()
        tas = (mev - cng_alt) / 1e6
        geri = yatirim_tl / (tas * 1e6) if tas > 0 else float('inf')
        senaryolar.append({'tuk_carpan': tuk_carpan, 'fiyat_carpan': fiyat_carpan, 'tasarruf_M_TL': tas, 'geri_odeme_yil': geri})

sens_df = pd.DataFrame(senaryolar)
print(sens_df.to_string(index=False))

# Worst case + best case
worst = sens_df.loc[sens_df['tasarruf_M_TL'].idxmin()]
best = sens_df.loc[sens_df['tasarruf_M_TL'].idxmax()]
print(f'\n=== SENSITIVITY SONUCU ===')
print(f'En kotumser senaryo: tasarruf={worst["tasarruf_M_TL"]:.2f}M TL/yıl, geri odeme={worst["geri_odeme_yil"]:.1f} yil')
print(f'En iyimser senaryo: tasarruf={best["tasarruf_M_TL"]:.2f}M TL/yıl, geri odeme={best["geri_odeme_yil"]:.1f} yil')
print(f'\nYatirim {yatirim_tl/1e6:.0f}M TL hangi senaryoda geri odemiyor?')
geri_odenmez = sens_df[sens_df['geri_odeme_yil'] > 15]
print(f'  Geri odeme >15 yil olan senaryolar: {len(geri_odenmez)} / {len(sens_df)}')


=== SENSITIVITY: SENARYO 1 (50 yasli motorin -> CNG) ===

 tuk_carpan  fiyat_carpan  tasarruf_M_TL  geri_odeme_yil
        0.8          0.85      11.269183       31.058151
        0.8          1.00      13.257863       26.399429
        0.8          1.15      15.246542       22.956025
        1.0          0.85      14.086479       24.846521
        1.0          1.00      16.572328       21.119543
        1.0          1.15      19.058178       18.364820
        1.2          0.85      16.903775       20.705434
        1.2          1.00      19.886794       17.599619
        1.2          1.15      22.869813       15.304017

=== SENSITIVITY SONUCU ===
En kotumser senaryo: tasarruf=11.27M TL/yıl, geri odeme=31.1 yil
En iyimser senaryo: tasarruf=22.87M TL/yıl, geri odeme=15.3 yil

Yatirim 350M TL hangi senaryoda geri odemiyor?
  Geri odeme >15 yil olan senaryolar: 9 / 9


---
## 12. KISIM D: Yenileme Oncelik Listesi (Synthesis)

Tek skorda birlestir: yas + arıza riski + yakit maliyeti -> oncelik skoru


In [12]:
# BOLUM 12: Yenileme Onceligi
# Normalize 3 boyut: yas, ort_skor, yıllık_yakit_tl
def norm(s):
    return ((s - s.min()) / (s.max() - s.min()) * 100).round(1)

arac_full['skor_yas'] = norm(arac_full['yas'])
arac_full['skor_ariza'] = norm(arac_full['ort_skor'])
arac_full['skor_yakit'] = norm(arac_full['yillik_yakit_tl'])

# Agirlikli skor: yas %40 + ariza %30 + yakit %30
arac_full['oncelik_skor'] = (arac_full['skor_yas']*0.4 + arac_full['skor_ariza']*0.3 + arac_full['skor_yakit']*0.3).round(1)

# Sadece motorin solo araclar yenileme adayi
adaylar = arac_full[(arac_full['YAKITTURU']=='MOTORIN') & (arac_full['ARACCINSI']=='SOLO')].copy()
top100 = adaylar.nlargest(100, 'oncelik_skor')

print('=== TOP 20 YENILEME ONCELIK LISTESI ===')
print(top100.head(20)[['KAPINO','GARAJ','MARKA','MODEL','yas','ort_skor','yillik_yakit_tl','oncelik_skor']].to_string(index=False))

print(f'\n=== TOP 100 OZETI ===')
print(f'Toplam yıllık yakıt maliyeti: {top100["yillik_yakit_tl"].sum()/1e6:.1f} milyon TL')
print(f'Ortalama yas: {top100["yas"].mean():.1f}')
print(f'Garaj dagilimi:')
print(top100['GARAJ'].value_counts().head(8))


=== TOP 20 YENILEME ONCELIK LISTESI ===
KAPINO     GARAJ    MARKA       MODEL  yas  ort_skor  yillik_yakit_tl  oncelik_skor
 M5508 Şahinkaya MERCEDES CITARO 0530 19.0  6.024444     1.057053e+06          67.6
 M6306 Şahinkaya MERCEDES CITARO 0530 19.0  5.697778     9.894907e+05          65.8
 M5740 Şahinkaya MERCEDES CITARO 0530 19.0  5.622000     1.018654e+06          65.7
 M5977 Şahinkaya MERCEDES CITARO 0530 19.0  5.740000     9.281736e+05          65.5
 M2739 Şahinkaya MERCEDES CITARO 0530 19.0  4.623000     1.495730e+06          65.0
 M3734 Şahinkaya MERCEDES CITARO 0530 19.0  5.240000     1.115082e+06          64.8
 M6303 Şahinkaya MERCEDES CITARO 0530 19.0  5.130000     1.097229e+06          64.3
 M3051 Şahinkaya MERCEDES CITARO 0530 19.0  5.283750     9.927393e+05          64.2
 M2187 Şahinkaya MERCEDES CITARO 0530 19.0  4.837778     1.233760e+06          64.1
 M5600 Şahinkaya MERCEDES CITARO 0530 19.0  4.920000     1.177451e+06          64.0
 M4036   Anadolu MERCEDES CITARO 053

---
## 13. ML V6 Feature Kandidatları

Bu analizden cikan ML feature'ları.


In [13]:
# BOLUM 13: ML Feature kandidatlari
print('=== ML V6 FEATURE KANDIDATI ===\n')
print('1. yakit_turu_cng: 0/1 binary')
print(f'   - r ile ciddi_ariza: {arac_yk["is_cng"].corr(arac_yk["ort_skor"]):.4f}')
print(f'   - Confounder altı (M4 katsayisi): {m4.params["is_cng"]:+.4f}')
print()
print('2. tahmini_yillik_yakit_tl: surekli')
sub = arac_full.dropna(subset=['yillik_yakit_tl','ort_skor'])
r_y = sub['yillik_yakit_tl'].corr(sub['ort_skor'])
print(f'   - r ile ort_skor: {r_y:.4f}')
print()
print('3. verimsizlik_skoru: yas_norm × tuketim_norm')
arac_full['verimsizlik_skor'] = (arac_full['skor_yas'] * arac_full['TUKETIM_BIRIM_100KM']/100).round(2)
r_v = arac_full['verimsizlik_skor'].corr(arac_full['ort_skor'])
print(f'   - r ile ort_skor: {r_v:.4f}')
print()
print('Notlar:')
print('  - yakit_turu_cng confounder altinda zayif (M4 k cok kucuk olabilir)')
print('  - tahmini_yillik_yakit_tl yas ile yuksek kolinearite olabilir')
print('  - Leakage testi ML_MODEL_V6 hazırlığında yapılacak')


=== ML V6 FEATURE KANDIDATI ===

1. yakit_turu_cng: 0/1 binary
   - r ile ciddi_ariza: 0.0355
   - Confounder altı (M4 katsayisi): -0.3575

2. tahmini_yillik_yakit_tl: surekli
   - r ile ort_skor: 0.0759

3. verimsizlik_skoru: yas_norm × tuketim_norm
   - r ile ort_skor: 0.2200

Notlar:
  - yakit_turu_cng confounder altinda zayif (M4 k cok kucuk olabilir)
  - tahmini_yillik_yakit_tl yas ile yuksek kolinearite olabilir
  - Leakage testi ML_MODEL_V6 hazırlığında yapılacak


---
## 14. Kısıtlamalar + Dis Veri Kaynak Listesi

Dürüstçe işaretlenmesi gereken metodolojik notlar.


In [14]:
# BOLUM 14: Kisitlamalar ve Kaynaklar
print('=== KISITLAMALAR ===\n')

kisitlamalar = [
    '1. Tuketim degerleri uretici spec + Istanbul BRT kalibrasyonu',
    '   - Gercek tuketim arac, surucu, yas, bakim durumuna gore varyans gosterir',
    '   - Aynı marka-modelin yeni vs yasli olanı arasinda %30+ fark olabilir',
    '',
    '2. Yakit fiyatlari aylık ortalama (EPDK)',
    '   - Bayi farkı, ihale fiyati avantaji (-%5-10) hesaba katilmadi',
    '   - CNG fiyat tahmini, sehirlere gore varyans var',
    '',
    '3. Arac alim fiyatlari Ankara 2024 ihalesinden ekstrapolasyon',
    '   - Istanbul İETT ihalesi gercek fiyatlar gizli, ±%15 hata payı',
    '',
    '4. 6 ay veriden yıllık km tahmini (×2)',
    '   - Mevsimsellik etkisi (yaz - kıs) yansitilmadi',
    '   - Bakim/arızalı gunler dusulmedi',
    '',
    '5. ARACTIPI ayrımı yapılmadı (proje kararı)',
    '   - Metrobus tüketimi otobusten farkli (BRT 60 L/100km gercek)',
    '   - Tuketim tablosunda ARACCINSI ile kismi ayrım var',
    '',
    '6. CNG arac M4 confounder altinda zayif sinyal verirse',
    '   - Filodaki CNG araclari sınırlı sayida, istatistiksel guc dusuk',
    '',
    '7. ROI hesabı sadece yakit tasarrufu uzerinden',
    '   - Bakim maliyeti azalmasi, emisyon vergisi avantaji eklenmemis',
]
for k in kisitlamalar:
    print(k)

print('\n=== DIS VERI KAYNAKLARI ===\n')
kaynaklar = [
    'Yakit fiyatlari: EPDK aylık ortalama bayi satıs (hakedis.org)',
    'CNG fiyat: BOTAS toptan tarife + belediye operatorleri',
    'Tuketim: BTS, Daimler Citaro Euro VI raporları + Belgrade CNG bus study',
    'Istanbul BRT kalibrasyonu: 78.4M km/yıl × 47M+ litre = 60 L/100km',
    'Arac fiyat: Ankara 2024 ihale (Otokar/Mercedes/BMC) + endustri std',
    'Vehicle specs: uretici resmi sayfalar (Otokar, BMC, Mercedes, Karsan, TEMSA)',
]
for k in kaynaklar:
    print(f'  - {k}')

print('\n=== SONRAKI ADIM ===')
print('SONUCLAR.md yazımı + Analiz 8 (Güvenlik Prioritizasyon) gecisi')


=== KISITLAMALAR ===

1. Tuketim degerleri uretici spec + Istanbul BRT kalibrasyonu
   - Gercek tuketim arac, surucu, yas, bakim durumuna gore varyans gosterir
   - Aynı marka-modelin yeni vs yasli olanı arasinda %30+ fark olabilir

2. Yakit fiyatlari aylık ortalama (EPDK)
   - Bayi farkı, ihale fiyati avantaji (-%5-10) hesaba katilmadi
   - CNG fiyat tahmini, sehirlere gore varyans var

3. Arac alim fiyatlari Ankara 2024 ihalesinden ekstrapolasyon
   - Istanbul İETT ihalesi gercek fiyatlar gizli, ±%15 hata payı

4. 6 ay veriden yıllık km tahmini (×2)
   - Mevsimsellik etkisi (yaz - kıs) yansitilmadi
   - Bakim/arızalı gunler dusulmedi

5. ARACTIPI ayrımı yapılmadı (proje kararı)
   - Metrobus tüketimi otobusten farkli (BRT 60 L/100km gercek)
   - Tuketim tablosunda ARACCINSI ile kismi ayrım var

6. CNG arac M4 confounder altinda zayif sinyal verirse
   - Filodaki CNG araclari sınırlı sayida, istatistiksel guc dusuk

7. ROI hesabı sadece yakit tasarrufu uzerinden
   - Bakim maliyeti az